## **Storing and Retrieving Vectors**

> This NoteBook demonstrates how to store and retrieve numerical vectors using a database. It uses:

- SQLite → lightweight database

- NumPy → for handling numerical vectors

- Binary storage (BLOB) → to store vectors efficiently

> This concept is widely used in:

- AI/ML systems

- Recommendation engines

- Semantic search (like ChatGPT embeddings)

## **Basic Concepts:**

1. Vector

A vector is simply a list of numbers:

> [1.2, 3.4, 2.1, 0.8]

Used in:

- Machine Learning

- Embeddings

- Similarity search

> 2. SQLite Database

- Lightweight database stored as a file (.db)

- No server required

- Supports SQL queries

> 3. BLOB (Binary Large Object)

- Used to store raw binary data

- Here we store vectors as byte streams

> 4. NumPy

- Library for numerical computing

- Helps convert vectors ↔ bytes

## **Importing**

In [2]:
# !pip install sqlite3
import sqlite3
import numpy as np

### **Creating Database Connection**

In [3]:
# create a connection to the SQLite DB
conn = sqlite3.connect('vector-db.db')
# Create a cursor object to execute SQL Commands
cursor = conn.cursor()  # Cusor -> USed to execute SQL commands

### **Creating table**

In [4]:
# Create a table for vector data
cursor.execute(
"""
CREATE TABLE IF NOT EXISTS vectors (
    id INTEGER PRIMARY KEY,
    vector BLOB NOT NULL
)
"""
)

In [ ]:
# generate some sample vectors
vect1 = np.array([1.2, 3.4, 2.1, 0.8])
vect2 = np.array([2.7, 1.5, 3.9, 2.3])

### **Convert Vector to bytes**

In [ ]:
vect1.tobytes() # numpy array to bytestream

b'333333\xf3?333333\x0b@\xcd\xcc\xcc\xcc\xcc\xcc\x00@\x9a\x99\x99\x99\x99\x99\xe9?'

In [ ]:
# Insert vector data into table
cursor.execute("INSERT INTO vectors (vector) VALUES (?)",
               (sqlite3.Binary(vect1.tobytes()),))


In [ ]:
cursor.execute("INSERT INTO vectors (vector) VALUES (?)",
               (sqlite3.Binary(vect2.tobytes()),))

In [ ]:
# Retreive data

cursor.execute("SELECT vector FROM vectors")

In [ ]:
rows = cursor.fetchall()

In [ ]:
rows

[(b'333333\xf3?333333\x0b@\xcd\xcc\xcc\xcc\xcc\xcc\x00@\x9a\x99\x99\x99\x99\x99\xe9?',),
 (b'\x9a\x99\x99\x99\x99\x99\x05@\x00\x00\x00\x00\x00\x00\xf8?333333\x0f@ffffff\x02@',)]

In [ ]:
vector = np.frombuffer(rows[0][0], dtype = np.float64)

In [ ]:
vector

array([1.2, 3.4, 2.1, 0.8])

In [ ]:
vectors = []
for row in rows:
    vector = np.frombuffer(row[0], dtype = np.float64)
    vectors.append(vector)

In [ ]:
vectors

[array([1.2, 3.4, 2.1, 0.8]), array([2.7, 1.5, 3.9, 2.3])]

# Vector Similarity Search (VSS)

In [ ]:
query_vect = np.array([1.0, 3.2, 2.0, 0.5])


In [ ]:
cursor.execute("""
SELECT vector FROM vectors ORDER BY abs(vector - ?) ASC
""", (sqlite3.Binary(query_vect.tobytes()),))

In [ ]:
res = cursor.fetchone()  # finding the top one

In [ ]:
np.frombuffer(res[0], dtype=np.float64) # most similar vector

array([2.7, 1.5, 3.9, 2.3])

In [ ]:
conn.commit()

In [ ]:
conn.close()